### Data integrity and leakage verifiaction

Primary goal for this notebook is to verify that the dataset we are using currently doesnt have data leakage. 

By running this notebook first, we establish a strict proof of data quality that guarantees:

1. **Document-Level Split Disjointness:** No text or notice used in evaluation/testing is visible to the models during training.


2. **Synthetic Data Isolation:** All synthetic training rows remain strictly inside the training set and are excluded from evaluation metrics.


3. **Persona Consistency:** Both the `Student` and `Faculty` persona views for a single document reside on the exact same side of the split.


4. **Reproducibility Audit:** An automated audit file (`week5_split_audit.json`) is generated to track cross-task split relationships.

We used assert which is a keyword used as a hard security check. which tells the program:

> *"This condition MUST be True. If it is False, stop execution immediately and throw an error!"*


If an `assert` condition is satisfied, Python silently continues to the next line. If it fails, execution halts instantly with an `AssertionError`. In our notebook, `assert` statements act as safety circuit-breakers

In [21]:
import json 
import hashlib
import warnings
from pathlib import Path
import pandas as pd 
import numpy as np 
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")


ROOT_DIR = Path.cwd()
while ROOT_DIR != ROOT_DIR.parent and not (ROOT_DIR / "pyproject.toml").exists() and not (ROOT_DIR / ".git").exists():
    ROOT_DIR = ROOT_DIR.parent

CLS_CSV   = ROOT_DIR / "data" / "processed" / "classification_dataset" / "classification_dataset.csv"
AUG_JSONL = ROOT_DIR / "data" / "summarizerdata" / "final_dataset_augmented.jsonl"
SPLIT_JSON = ROOT_DIR / "submission" / "week_4" / "summarizer_split.json"
WEEK5_DIR = ROOT_DIR / "submission" / "week_5"

if not CLS_CSV.is_file():
    raise FileNotFoundError(f"Classification dataset not found at expected path: {CLS_CSV}")

print("Classification CSV :", CLS_CSV)
print("Summarizer Corpus  :", AUG_JSONL)
print("Summarizer Split   :", SPLIT_JSON)
print('all works')

Classification CSV : c:\ProjectFiles\uni-comms-intelligence\data\processed\classification_dataset\classification_dataset.csv
Summarizer Corpus  : c:\ProjectFiles\uni-comms-intelligence\data\summarizerdata\final_dataset_augmented.jsonl
Summarizer Split   : c:\ProjectFiles\uni-comms-intelligence\submission\week_4\summarizer_split.json
all works


In [22]:
# loading the classification dataset 
df=pd.read_csv(CLS_CSV)
df["document_id"] = df["document_id"].astype(str)
df["split"] = df["split"].replace({"validation": "val"})
df["is_synthetic"] = df["is_synthetic"].astype(bool)

counts = df.groupby(["split", "is_synthetic"]).size().unstack(fill_value=0)
counts = counts.rename(columns={False: "real", True: "synthetic"})
counts["total"] = counts.sum(axis=1)



In [23]:
print("Classification Dataset Split Counts")
display(counts)

expected = {"total_rows": 150, "real": 120, "synthetic": 30, "train": 108, "val": 18, "test": 24}
actual = {
    "total_rows": len(df),
    "real": int((~df.is_synthetic).sum()),
    "synthetic": int(df.is_synthetic.sum()),
    **{s: int((df.split == s).sum()) for s in ["train", "val", "test"]}
}

chk = pd.DataFrame({"expected": expected, "actual": actual})
chk["match"] = chk["expected"] == chk["actual"]
display(chk)

Classification Dataset Split Counts


is_synthetic,real,synthetic,total
split,,,
test,24,0,24
train,78,30,108
val,18,0,18


,expected,actual,match
total_rows,150,150,True
real,120,120,True
synthetic,30,30,True
train,108,108,True
val,18,18,True
test,24,24,True


In [24]:
ids = {s: set(df.loc[df.split == s, "document_id"]) for s in ["train", "val", "test"]}

# 1. One row per document ID
assert df["document_id"].is_unique, "LEAKAGE ERROR: A document_id appears in multiple rows/splits!"

# 2. Document-level disjointness across train, val, and test
assert ids["train"].isdisjoint(ids["val"]), "LEAKAGE ERROR: Train and Validation sets overlap!"
assert ids["train"].isdisjoint(ids["test"]), "LEAKAGE ERROR: Train and Test sets overlap!"
assert ids["val"].isdisjoint(ids["test"]), "LEAKAGE ERROR: Validation and Test sets overlap!"
print("OK: Document-level splits are strictly disjoint:", {k: len(v) for k, v in ids.items()})

# 3. Synthetic rows are strictly train-only
bad_synth = df[(df.split != "train") & df.is_synthetic]
assert bad_synth.empty, f"LEAKAGE ERROR: {len(bad_synth)} synthetic rows found outside train!"
print(f"OK: All {int(df.is_synthetic.sum())} synthetic rows are isolated in train set.")

# 4. Check for exact duplicate cleaned text across splits
df["text_hash"] = df["cleaned_text"].fillna("").map(lambda t: hashlib.md5(" ".join(t.split()).lower().encode()).hexdigest())
dup = df.groupby("text_hash")["split"].nunique()
cross_dups = dup[dup > 1]
assert cross_dups.empty, f"LEAKAGE ERROR: {len(cross_dups)} identical texts found across splits!"
print("OK: Zero identical texts shared across splits.")

OK: Document-level splits are strictly disjoint: {'train': 108, 'val': 18, 'test': 24}
OK: All 30 synthetic rows are isolated in train set.
OK: Zero identical texts shared across splits.


In [25]:
SIM_THRESHOLD = 0.90

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words="english", sublinear_tf=True)
vec.fit(df["cleaned_text"].fillna(""))

M = {s: vec.transform(df.loc[df.split == s, "cleaned_text"].fillna("")) for s in ["train", "val", "test"]}
train_df = df[df.split == "train"].reset_index(drop=True)
syn_mask = train_df.is_synthetic.values

rows = []
for s in ["val", "test"]:
    sim = (M[s] @ M["train"].T).toarray()
    sub = df[df.split == s].reset_index(drop=True)
    for i in range(sim.shape[0]):
        real_max = sim[i, ~syn_mask].max() if (~syn_mask).any() else 0.0
        syn_max = sim[i, syn_mask].max() if syn_mask.any() else 0.0
        rows.append({
            "split": s,
            "document_id": sub.loc[i, "document_id"],
            "max_sim_to_real_train": round(float(real_max), 3),
            "max_sim_to_synthetic_train": round(float(syn_max), 3)
        })

near_dups = pd.DataFrame(rows)
flagged = near_dups[(near_dups.max_sim_to_real_train > SIM_THRESHOLD) | (near_dups.max_sim_to_synthetic_train > SIM_THRESHOLD)]

print(f"Scan Result: {len(flagged)} of {len(near_dups)} val/test documents exceed cosine {SIM_THRESHOLD} with train docs.")
display(flagged if len(flagged) else pd.DataFrame({"result": ["No near-duplicates above threshold"]}))

Scan Result: 2 of 42 val/test documents exceed cosine 0.9 with train docs.


,split,document_id,max_sim_to_real_train,max_sim_to_synthetic_train
36,test,94,0.904,0.039
41,test,121,0.906,0.033


In [26]:
# Load augmented summarizer corpus and split file
aug = [json.loads(line) for line in open(AUG_JSONL, encoding="utf-8") if line.strip()]
all_doc_ids = {str(a["document_id"]) for a in aug}

split_data = json.loads(SPLIT_JSON.read_text())
summ_train = set(map(str, split_data.get("train_ids", split_data.get("train_doc_ids", []))))
summ_eval = set(map(str, split_data.get("eval_ids", split_data.get("test_doc_ids", []))))

# Assert summarizer split disjointness
assert summ_train.isdisjoint(summ_eval), "LEAKAGE ERROR: Summarizer train and eval IDs overlap!"
assert summ_train | summ_eval == all_doc_ids, "ERROR: Split does not cover all 75 summarizer notices!"
print(f"OK: Summarizer split: {len(summ_train)} Train Docs / {len(summ_eval)} Eval Docs (75 Total).")

# Verify both Student and Faculty rows for each document sit on the same split side
rows = []
for a in aug:
    doc_id = str(a["document_id"])
    side = "train" if doc_id in summ_train else "eval"
    for persona in ("student", "faculty"):
        rows.append({"doc_id": doc_id, "persona": persona, "side": side})

pr = pd.DataFrame(rows)
sides_per_doc = pr.groupby("doc_id")["side"].nunique()
assert (sides_per_doc == 1).all(), "LEAKAGE ERROR: Student and Faculty views of a notice are on different splits!"
print("OK: Persona views consistent:", pr.groupby("side").size().to_dict(), "rows (train/eval).")

OK: Summarizer split: 60 Train Docs / 15 Eval Docs (75 Total).
OK: Persona views consistent: {'eval': 30, 'train': 120} rows (train/eval).


In [27]:
cls_split = df.set_index("document_id")["split"]
rel = pd.DataFrame({"doc_id": sorted(summ_eval)})
rel["classification_split"] = rel.doc_id.map(cls_split).fillna("not in classification csv")

e2e_ids = sorted(rel.loc[rel.classification_split.isin(["test", "val"]), "doc_id"], key=lambda x: (len(x), x))
e2e_test_only = sorted(rel.loc[rel.classification_split == "test", "doc_id"], key=lambda x: (len(x), x))

# Save audit object
audit_data = {
    "classification_counts": {k: int(v) for k, v in actual.items()},
    "classification_split_disjoint": True,
    "synthetic_train_only": True,
    "no_identical_text_across_splits": True,
    "near_duplicate_threshold": SIM_THRESHOLD,
    "near_duplicate_flagged": flagged.to_dict("records"),
    "summarizer_train_docs": len(summ_train),
    "summarizer_eval_docs": len(summ_eval),
    "summarizer_persona_rows_consistent": True,
    "summarizer_eval_by_classification_split": rel.classification_split.value_counts().to_dict(),
    "e2e_heldout_ids_val_or_test": e2e_ids,
    "e2e_heldout_ids_test_only": e2e_test_only,
    "summarizer_eval_ids": sorted(summ_eval, key=lambda x: (len(x), x)),
    "summarizer_train_ids": sorted(summ_train, key=lambda x: (len(x), x)),
}

audit_path = WEEK5_DIR / "week5_split_audit.json"
audit_path.write_text(json.dumps(audit_data, indent=2))

print(f"Saved split audit report to {audit_path}")

Saved split audit report to c:\ProjectFiles\uni-comms-intelligence\submission\week_5\week5_split_audit.json
